# Homework 2: candidate generators + ML reranker

This notebook builds a treatment candidate source for `botify` without changing the online Docker dependencies.

The pipeline is:

1. Read `botify` logs collected from simulator runs.
2. Recover training examples of the form `(session context, candidate track) -> next listen time`.
3. Build several candidate sources: transition-I2I, LightFM-I2I, content nearest neighbours, and global good tracks.
4. Train a CatBoost classifier if available, otherwise use a sklearn fallback.
5. Export a ranked I2I file compatible with `I2IRecommender`.


In [1]:
import glob
import json
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from catboost import CatBoostClassifier, Pool

RANDOM_STATE = 31337
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


catboost available: True


## Config

Set `LOG_DIRS` to folders created by `script/dataclient.py ... log2local ...`.


In [2]:
REPO_DIR = Path("..").resolve()

# Edit this list for your collected simulator logs.
LOG_DIRS = [
    REPO_DIR / "data" / "raw_run_01",
    REPO_DIR / "data" / "catboost",
    REPO_DIR / "data" / "catboost2",
    REPO_DIR / "data" / "training_data",
]

TRACKS_PATH = REPO_DIR / "botify" / "data" / "tracks.json"
OUTPUT_I2I_PATH = REPO_DIR / "botify" / "data" / "ranker_i2i.jsonl"
LIGHTFM_I2I_PATH = REPO_DIR / "botify" / "data" / "lightfm_i2i.jsonl"

GOOD_TIME_THRESHOLD = 0.7
HISTORY_LIMIT = 10
NEGATIVES_PER_POSITIVE = 5
MAX_POSITIVE_EXAMPLES = 500_000

TRANSITION_K = 120
CONTENT_K = 120
LIGHTFM_K = 120
POPULAR_K = 200
EXPORT_ITEMS = None  # None means export all tracks from the catalog
EXPORT_K = 50

print("repo:", REPO_DIR)
print("tracks:", TRACKS_PATH)
print("output:", OUTPUT_I2I_PATH)

repo: C:\Users\Егор\Desktop\vk-course\recsys\recsys-course-spring-2026
tracks: C:\Users\Егор\Desktop\vk-course\recsys\recsys-course-spring-2026\botify\data\tracks.json
output: C:\Users\Егор\Desktop\vk-course\recsys\recsys-course-spring-2026\botify\data\ranker_i2i.jsonl


## Read logs



In [3]:
def read_logs(log_dirs):
    paths = []
    for log_dir in log_dirs:
        paths.extend(glob.glob(str(Path(log_dir) / "**" / "data.json*"), recursive=True))
    paths = sorted(set(paths))
    if not paths:
        raise FileNotFoundError(f"No data.json files found in {log_dirs}")
    print("log files:", len(paths))
    data = pd.concat([pd.read_json(path, lines=True) for path in paths], ignore_index=True)
    data = data[data["message"].isin(["next", "last"])].copy()
    data = data.dropna(subset=["user", "track", "timestamp", "time"])
    data["user"] = data["user"].astype("int32")
    data["track"] = data["track"].astype("int32")
    data["time"] = data["time"].astype("float32")
    data["timestamp"] = data["timestamp"].astype("int64")
    data = data.sort_values(["user", "timestamp"]).reset_index(drop=True)
    return data


logs = read_logs(LOG_DIRS)
print(logs.shape)
logs.head()

log files: 8
(864958, 8)


,message,timestamp,user,track,time,latency,recommendation,experiments
0,next,1777099500718000000,0,2020,1.00,0.001186,4405.0,{'HSTU': 'C'}
1,next,1777099500766000000,0,4405,0.64,0.001212,9214.0,{'HSTU': 'C'}
2,next,1777099500814000000,0,9214,0.61,0.001571,9465.0,{'HSTU': 'C'}
3,next,1777099500867000000,0,9465,0.86,0.001850,9435.0,{'HSTU': 'C'}
4,next,1777099500914000000,0,9435,0.87,0.001354,9442.0,{'HSTU': 'C'}


## Track content vectors



In [4]:
tracks = pd.read_json(TRACKS_PATH, lines=True).drop_duplicates("track").sort_values("track").reset_index(drop=True)
tracks["genres_text"] = tracks["genres"].map(lambda x: " ".join(x) if isinstance(x, list) else "")
tracks["artist_genres_text"] = tracks["artist_genres"].map(lambda x: " ".join(x) if isinstance(x, list) else "")
tracks["text"] = (
    tracks["title"].fillna("") + " " +
    tracks["artist"].fillna("") + " " +
    tracks["genres_text"] + " " +
    tracks["artist_genres_text"] + " " +
    tracks["mood"].fillna("") + " " +
    tracks["artist_country"].fillna("") + " " +
    tracks["summary"].fillna("")
)

track_ids = tracks["track"].astype(int).to_numpy()
track_to_pos = {track: pos for pos, track in enumerate(track_ids)}

tfidf = TfidfVectorizer(max_features=8192, min_df=2, ngram_range=(1, 2))
x_text = tfidf.fit_transform(tracks["text"])

n_components = min(128, x_text.shape[1] - 1)
svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)
track_vectors = svd.fit_transform(x_text).astype("float32")
track_vectors = normalize(track_vectors).astype("float32")

track_artist = tracks.set_index("track")["artist"].to_dict()
track_genre = tracks.set_index("track")["artist_genre"].to_dict()

# Some catalog rows contain year ranges like "1941-1942".
# Use the first 4-digit year and keep missing/malformed values as 0.
tracks["year_num"] = (
    tracks["year"]
    .astype("string")
    .str.extract(r"(\d{4})", expand=False)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype("float32")
)
tracks["artist_fans_num"] = pd.to_numeric(tracks["artist_fans"], errors="coerce").fillna(0).astype("float32")

track_year = tracks.set_index("track")["year_num"].to_dict()
track_fans = tracks.set_index("track")["artist_fans_num"].to_dict()

print("tracks:", len(tracks))
print("text matrix:", x_text.shape)
print("vectors:", track_vectors.shape)
tracks.head(2)

tracks: 16198
text matrix: (16198, 8192)
vectors: (16198, 128)


,title,alternative_title,artist,alternative_artist,genres,year,mood,summary,artist_id,artist_country,artist_genres,artist_genre,artist_fans,track,genres_text,artist_genres_text,text,year_num,artist_fans_num
0,Dancing Queen,None,ABBA,None,"[Pop, Disco]",1976,Nostalgic,The song tells the story of a woman who realiz...,0,Sweden,"[Pop, Disco]",Pop,100.0,0,Pop Disco,Pop Disco,Dancing Queen ABBA Pop Disco Pop Disco Nostalg...,1976.0,100.0
1,Mamma Mia,None,ABBA,None,"[Pop, Disco]",1975,Energetic,The song's lyrics are about a woman who is try...,0,Sweden,"[Pop, Disco]",Pop,100.0,1,Pop Disco,Pop Disco,Mamma Mia ABBA Pop Disco Pop Disco Energetic S...,1975.0,100.0


## Recover served examples



In [5]:
def build_served_examples(df, history_limit=10):
    examples = []
    cols = ["message", "timestamp", "user", "track", "time", "recommendation"]
    for user, group in tqdm(df[cols].groupby("user", sort=False), desc="users"):
        session_tracks = []
        session_times = []
        rows = list(group.sort_values("timestamp").itertuples(index=False))
        for idx, row in enumerate(rows):
            current_track = int(row.track)
            current_time = float(row.time)
            session_tracks.append(current_track)
            session_times.append(current_time)

            if row.message == "next" and pd.notna(row.recommendation) and idx + 1 < len(rows):
                candidate = int(row.recommendation)
                next_row = rows[idx + 1]
                if int(next_row.track) == candidate:
                    examples.append({
                        "user": int(user),
                        "timestamp": int(row.timestamp),
                        "prev_track": current_track,
                        "prev_time": current_time,
                        "candidate": candidate,
                        "target_time": float(next_row.time),
                        "history_tracks": tuple(session_tracks[-history_limit:]),
                        "history_times": tuple(session_times[-history_limit:]),
                    })

            if row.message == "last":
                session_tracks = []
                session_times = []

    return pd.DataFrame(examples)


served = build_served_examples(logs, HISTORY_LIMIT)
served["target"] = (served["target_time"] >= GOOD_TIME_THRESHOLD).astype("int8")
print(served.shape)
served.head()

users:   0%|          | 0/10000 [00:00<?, ?it/s]

(609497, 9)


,user,timestamp,prev_track,prev_time,candidate,target_time,history_tracks,history_times,target
0,0,1777099500718000000,2020,1.00,4405,0.64,"(2020,)","(1.0,)",0
1,0,1777099500766000000,4405,0.64,9214,0.61,"(2020, 4405)","(1.0, 0.6399999856948853)",0
2,0,1777099500814000000,9214,0.61,9465,0.86,"(2020, 4405, 9214)","(1.0, 0.6399999856948853, 0.6100000143051147)",1
3,0,1777099500867000000,9465,0.86,9435,0.87,"(2020, 4405, 9214, 9465)","(1.0, 0.6399999856948853, 0.6100000143051147, ...",1
4,0,1777099500914000000,9435,0.87,9442,0.27,"(2020, 4405, 9214, 9465, 9435)","(1.0, 0.6399999856948853, 0.6100000143051147, ...",0


## Candidate sources



In [6]:
def read_i2i_jsonl(path, k=None):
    result = {}
    if not Path(path).exists():
        print("missing i2i file:", path)
        return result
    with open(path, encoding="utf-8") as i2i_file:
        for line in i2i_file:
            row = json.loads(line)
            recs = [int(x) for x in row["recommendations"]]
            result[int(row["item_id"])] = recs if k is None else recs[:k]
    return result


lightfm_candidates = read_i2i_jsonl(LIGHTFM_I2I_PATH, LIGHTFM_K)
lightfm_rank = {
    (anchor, candidate): rank + 1
    for anchor, recs in lightfm_candidates.items()
    for rank, candidate in enumerate(recs)
}

track_stats = (
    logs.groupby("track")["time"]
    .agg(track_count="size", track_mean="mean", track_sum="sum")
    .reset_index()
)
track_count = track_stats.set_index("track")["track_count"].to_dict()
track_mean = track_stats.set_index("track")["track_mean"].to_dict()

global_good_tracks = (
    track_stats.assign(score=lambda x: x["track_mean"] * np.log1p(x["track_count"]))
    .sort_values("score", ascending=False)["track"]
    .astype(int)
    .head(POPULAR_K)
    .tolist()
)

transition_stats = (
    served.groupby(["prev_track", "candidate"])["target_time"]
    .agg(trans_count="size", trans_mean="mean", trans_sum="sum")
    .reset_index()
)
transition_stats["trans_score"] = transition_stats["trans_mean"] * np.log1p(transition_stats["trans_count"])

transition_candidates = defaultdict(list)
transition_feature = {}
for prev_track, group in transition_stats.sort_values("trans_score", ascending=False).groupby("prev_track"):
    recs = group.head(TRANSITION_K)
    transition_candidates[int(prev_track)] = recs["candidate"].astype(int).tolist()
    for row in recs.itertuples(index=False):
        transition_feature[(int(row.prev_track), int(row.candidate))] = (
            float(row.trans_score), float(row.trans_mean), int(row.trans_count)
        )

nn = NearestNeighbors(n_neighbors=CONTENT_K + 1, metric="cosine", algorithm="brute")
nn.fit(track_vectors)
distances, indices = nn.kneighbors(track_vectors, return_distance=True)

content_candidates = {}
content_rank = {}
for pos, neighbours in enumerate(indices):
    source_track = int(track_ids[pos])
    recs = []
    for rank, neighbour_pos in enumerate(neighbours):
        candidate = int(track_ids[neighbour_pos])
        if candidate == source_track:
            continue
        recs.append(candidate)
        content_rank[(source_track, candidate)] = len(recs)
        if len(recs) >= CONTENT_K:
            break
    content_candidates[source_track] = recs

popular_rank = {track: rank + 1 for rank, track in enumerate(global_good_tracks)}

print("lightfm anchors:", len(lightfm_candidates))
print("transition anchors:", len(transition_candidates))
print("content anchors:", len(content_candidates))
print("global candidates:", len(global_good_tracks))

lightfm anchors: 15000
transition anchors: 16197
content anchors: 16198
global candidates: 200


## Feature builder


In [7]:
def get_vector(track):
    pos = track_to_pos.get(int(track))
    if pos is None:
        return None
    return track_vectors[pos]


def build_session_profile(history_tracks, history_times):
    vectors = []
    weights = []
    for track, time_value in zip(history_tracks, history_times):
        vector = get_vector(track)
        if vector is not None:
            vectors.append(vector)
            weights.append(max(float(time_value), 0.05))
    if not vectors:
        return None
    profile = np.average(np.vstack(vectors), axis=0, weights=np.asarray(weights, dtype="float32"))
    norm = np.linalg.norm(profile)
    return profile / norm if norm > 0 else profile


def candidate_pool(prev_track):
    pool = []
    pool.extend(transition_candidates.get(int(prev_track), []))
    pool.extend(lightfm_candidates.get(int(prev_track), []))
    pool.extend(content_candidates.get(int(prev_track), []))
    pool.extend(global_good_tracks)
    return list(dict.fromkeys(int(track) for track in pool))


def make_features(prev_track, prev_time, candidate, history_tracks, history_times):
    prev_track = int(prev_track)
    candidate = int(candidate)
    history_tracks = tuple(int(track) for track in history_tracks)
    history_times = tuple(float(time_value) for time_value in history_times)

    prev_vector = get_vector(prev_track)
    candidate_vector = get_vector(candidate)
    profile = build_session_profile(history_tracks, history_times)

    trans_score, trans_mean, trans_count = transition_feature.get((prev_track, candidate), (0.0, 0.0, 0))
    artists = [track_artist.get(track) for track in history_tracks]
    artist_counter = Counter(artists)

    candidate_artist = track_artist.get(candidate)
    prev_artist = track_artist.get(prev_track)
    candidate_genre = track_genre.get(candidate)
    prev_genre = track_genre.get(prev_track)

    cos_prev = 0.0
    cos_session = 0.0
    if prev_vector is not None and candidate_vector is not None:
        cos_prev = float(np.dot(prev_vector, candidate_vector))
    if profile is not None and candidate_vector is not None:
        cos_session = float(np.dot(profile, candidate_vector))

    return {
        "prev_time": float(prev_time),
        "history_len": len(history_tracks),
        "candidate_seen": int(candidate in set(history_tracks)),
        "same_artist_as_prev": int(candidate_artist == prev_artist),
        "same_genre_as_prev": int(candidate_genre == prev_genre),
        "artist_repeat_count": int(artist_counter.get(candidate_artist, 0)),
        "cos_prev": cos_prev,
        "cos_session": cos_session,
        "transition_score": float(trans_score),
        "transition_mean": float(trans_mean),
        "transition_count": int(trans_count),
        "lightfm_rank": float(lightfm_rank.get((prev_track, candidate), LIGHTFM_K + 100)),
        "is_from_lightfm": int((prev_track, candidate) in lightfm_rank),
        "content_rank": float(content_rank.get((prev_track, candidate), CONTENT_K + 100)),
        "popular_rank": float(popular_rank.get(candidate, POPULAR_K + 100)),
        "track_count": float(track_count.get(candidate, 0)),
        "track_mean": float(track_mean.get(candidate, 0.0)),
        "candidate_year": float(track_year.get(candidate, 0) or 0),
        "candidate_fans": float(track_fans.get(candidate, 0) or 0),
    }


feature_columns = list(make_features(0, 1.0, 1, (0,), (1.0,)).keys())
feature_columns

['prev_time',
 'history_len',
 'candidate_seen',
 'same_artist_as_prev',
 'same_genre_as_prev',
 'artist_repeat_count',
 'cos_prev',
 'cos_session',
 'transition_score',
 'transition_mean',
 'transition_count',
 'lightfm_rank',
 'is_from_lightfm',
 'content_rank',
 'popular_rank',
 'track_count',
 'track_mean',
 'candidate_year',
 'candidate_fans']

## Build ranker dataset



In [8]:
rng = np.random.default_rng(RANDOM_STATE)
positive = served.copy()
if len(positive) > MAX_POSITIVE_EXAMPLES:
    positive = positive.sample(MAX_POSITIVE_EXAMPLES, random_state=RANDOM_STATE).reset_index(drop=True)

rows = []
for row in tqdm(positive.itertuples(index=False), total=len(positive), desc="examples"):
    rows.append({
        **make_features(row.prev_track, row.prev_time, row.candidate, row.history_tracks, row.history_times),
        "target": int(row.target),
        "target_time": float(row.target_time),
        "is_observed": 1,
        "weight": 1.0,
    })

    pool = [track for track in candidate_pool(row.prev_track) if track != int(row.candidate)]
    seen = set(row.history_tracks)
    pool = [track for track in pool if track not in seen]
    if not pool:
        continue
    sample_size = min(NEGATIVES_PER_POSITIVE, len(pool))
    for candidate in rng.choice(pool, size=sample_size, replace=False):
        rows.append({
            **make_features(row.prev_track, row.prev_time, int(candidate), row.history_tracks, row.history_times),
            "target": 0,
            "target_time": 0.0,
            "is_observed": 0,
            "weight": 0.2,
        })

ranker_data = pd.DataFrame(rows)
print(ranker_data.shape)
print(ranker_data["target"].mean())
ranker_data.head()

examples:   0%|          | 0/500000 [00:00<?, ?it/s]

(3000000, 23)
0.04594566666666667


,prev_time,history_len,candidate_seen,same_artist_as_prev,same_genre_as_prev,artist_repeat_count,cos_prev,cos_session,transition_score,transition_mean,...,content_rank,popular_rank,track_count,track_mean,candidate_year,candidate_fans,target,target_time,is_observed,weight
0,1.0,1,0,0,0,0,0.091815,0.091816,0.013863,0.02,...,220.0,300.0,93.0,0.616667,2006.0,10.0,0,0.02,1,1.0
1,1.0,1,0,0,0,0,0.234741,0.234741,0.000000,0.00,...,220.0,34.0,260.0,0.754923,2009.0,75.0,0,0.00,0,0.2
2,1.0,1,0,0,1,0,0.461003,0.461003,0.000000,0.00,...,220.0,137.0,137.0,0.767299,2013.0,20.0,0,0.00,0,0.2
3,1.0,1,0,0,0,0,0.344060,0.344060,0.637695,0.92,...,220.0,300.0,117.0,0.715214,2010.0,100.0,0,0.00,0,0.2
4,1.0,1,0,0,0,0,0.523942,0.523942,0.000000,0.00,...,48.0,300.0,312.0,0.572660,2008.0,100.0,0,0.00,0,0.2


## Train reranker

CatBoost is preferred. If it is not installed, the notebook falls back to `HistGradientBoostingClassifier`, which is already available in sklearn.


In [9]:
train_df, valid_df = train_test_split(
    ranker_data,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=ranker_data["target"],
)

X_train = train_df[feature_columns]
y_train = train_df["target"].astype(int)
X_valid = valid_df[feature_columns]
y_valid = valid_df["target"].astype(int)
w_train = train_df["weight"].astype(float)
w_valid = valid_df["weight"].astype(float)

model.fit(X_train, y_train, sample_weight=w_train)
valid_score = model.predict_proba(X_valid)[:, 1]
train_pool = Pool(X_train, y_train, weight=w_train)
valid_pool = Pool(X_valid, y_valid, weight=w_valid)
model = CatBoostClassifier(
    iterations=1500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=100,
)
model.fit(train_pool, eval_set=valid_pool, use_best_model=True)
valid_score = model.predict_proba(X_valid)[:, 1]

print("roc_auc:", roc_auc_score(y_valid, valid_score))
print("average_precision:", average_precision_score(y_valid, valid_score))

0:	test: 0.9767686	best: 0.9767686 (0)	total: 350ms	remaining: 8m 44s
100:	test: 0.9856058	best: 0.9856058 (100)	total: 17.5s	remaining: 4m 2s
200:	test: 0.9865463	best: 0.9865463 (200)	total: 34s	remaining: 3m 40s
300:	test: 0.9869310	best: 0.9869310 (300)	total: 50.2s	remaining: 3m 19s
400:	test: 0.9871574	best: 0.9871574 (400)	total: 1m 6s	remaining: 3m 1s
500:	test: 0.9872930	best: 0.9872930 (500)	total: 1m 22s	remaining: 2m 44s
600:	test: 0.9873857	best: 0.9873857 (600)	total: 1m 41s	remaining: 2m 31s
700:	test: 0.9874543	best: 0.9874543 (700)	total: 1m 59s	remaining: 2m 16s
800:	test: 0.9875039	best: 0.9875039 (800)	total: 2m 12s	remaining: 1m 55s
900:	test: 0.9875454	best: 0.9875456 (899)	total: 2m 26s	remaining: 1m 37s
1000:	test: 0.9875821	best: 0.9875821 (1000)	total: 2m 40s	remaining: 1m 19s
1100:	test: 0.9876110	best: 0.9876113 (1098)	total: 2m 54s	remaining: 1m 3s
1200:	test: 0.9876340	best: 0.9876344 (1196)	total: 3m 9s	remaining: 47.2s
1300:	test: 0.9876534	best: 0.98765

## Export ranked I2I



In [10]:
def predict_scores(frame):
    if HAS_CATBOOST:
        return model.predict_proba(frame[feature_columns])[:, 1]
    return model.predict_proba(frame[feature_columns])[:, 1]


export_item_ids = (
    pd.Series(track_ids)
    .to_frame("track")
    .merge(track_stats[["track", "track_count"]], on="track", how="left")
    .fillna({"track_count": 0})
    .sort_values("track_count", ascending=False)["track"]
    .astype(int)
    .tolist()
)
if EXPORT_ITEMS is not None:
    export_item_ids = export_item_ids[:EXPORT_ITEMS]
print("export anchors:", len(export_item_ids))

OUTPUT_I2I_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_I2I_PATH, "w", encoding="utf-8") as out:
    for item_id in tqdm(export_item_ids, desc="export i2i"):
        pool = [track for track in candidate_pool(item_id) if track != item_id]
        pool = list(dict.fromkeys(pool))
        if not pool:
            pool = [track for track in global_good_tracks if track != item_id]

        features = pd.DataFrame([
            make_features(item_id, 1.0, candidate, (item_id,), (1.0,))
            for candidate in pool
        ])
        features["candidate"] = pool
        features["score"] = predict_scores(features)

        recommendations = (
            features.sort_values("score", ascending=False)["candidate"]
            .astype(int)
            .head(EXPORT_K)
            .tolist()
        )
        out.write(json.dumps({
            "item_id": int(item_id),
            "recommendations": recommendations,
        }) + "\n")

print("saved:", OUTPUT_I2I_PATH)

export anchors: 16198


export i2i:   0%|          | 0/16198 [00:00<?, ?it/s]

saved: C:\Users\Егор\Desktop\vk-course\recsys\recsys-course-spring-2026\botify\data\ranker_i2i.jsonl
